<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/05_applications/advanced_semantic_search_chatbot_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Semantic Search Chatbot Pipeline

## Objective

Build a robust semantic search chatbot that retrieves candidate
documents using embeddings, re-ranks them using a cross-encoder,
and applies confidence checks before returning the final answer.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

In [ ]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
documents = [
    "Machine learning tutorials help beginners understand ML concepts.",
    "Deep learning uses neural networks to learn complex patterns.",
    "Natural language processing focuses on understanding text data.",
    "Python is widely used for machine learning and data science.",
    "Transformers are powerful models used in modern NLP systems."
]

In [ ]:
doc_embeddings = bi_encoder.encode(documents)

In [ ]:
query = "How can I start learning machine learning?"
query_embedding = bi_encoder.encode(query)

In [10]:
scores = cosine_similarity([query_embedding], doc_embeddings)[0]

df = pd.DataFrame({
    "Document": documents,
    "Embedding Score": np.round(scores,4)
}).sort_values(by="Embedding Score", ascending=False)

top_k = 3
candidates = df.head(top_k).copy()

In [11]:
pairs = [[query, doc] for doc in candidates["Document"]]

rerank_scores = cross_encoder.predict(pairs)

candidates["CrossEncoder Score"] = np.round(rerank_scores,4)

candidates = candidates.sort_values(
    by="CrossEncoder Score",
    ascending=False
)

In [12]:
top_score = candidates.iloc[0]["CrossEncoder Score"]
second_score = candidates.iloc[1]["CrossEncoder Score"]

gap = top_score - second_score

confidence = "High" if gap > 1 else "Low"

In [13]:
print("User:", query)

if confidence == "High":
    print("Bot:", candidates.iloc[0]["Document"])
else:
    print("Bot: I'm not confident about the answer in my knowledge base.")

User: How can I start learning machine learning?
Bot: Machine learning tutorials help beginners understand ML concepts.
